# 盐湖补水溶矿：高分辨率优势通道结果

本 Notebook 只回答一个问题：与固定介质相比，矿物溶解引起的孔隙率—水力传导系数反馈，是否使水流更集中并使补给水更早到达采卤井？

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display


def locate_case_dir():
    start = Path.cwd().resolve()
    for parent in (start, *start.parents):
        if parent.name == "SaltLake_Brine3D" and (parent / "run.py").is_file():
            return parent
        candidate = parent / "examples" / "SaltLake_Brine3D"
        if (candidate / "run.py").is_file():
            return candidate
    raise FileNotFoundError("无法定位 examples/SaltLake_Brine3D")


CASE_DIR = locate_case_dir()
PROFILE = os.getenv("SALT_LAKE_PROFILE", "highres").strip().lower()
for scenario in ("feedback", "fixed"):
    result_path = CASE_DIR / "output" / f"{PROFILE}_{scenario}" / "results.npy"
    if not result_path.is_file():
        raise FileNotFoundError(f"缺少 {result_path}；请先完成 {PROFILE}_{scenario} 计算。")

diagnostic = subprocess.run(
    [
        sys.executable,
        str(CASE_DIR / "plot.py"),
        "--channels",
        "--profile",
        PROFILE,
    ],
    cwd=CASE_DIR.parent.parent,
    check=True,
    capture_output=True,
    text=True,
)
COMPARISON_DIR = CASE_DIR / "output" / f"{PROFILE}_channel_comparison"
summary = json.loads((COMPARISON_DIR / "channel_summary.json").read_text(encoding="utf-8"))
print(f"读取情景：{PROFILE}_feedback 与 {PROFILE}_fixed")
print(f"图件目录：{COMPARISON_DIR}")

## 结论表

判据很直接：feedback 相对 fixed 应表现为更早突破、更窄的有效过流宽度、更高的高流速单元份额，以及空间上更早的补给水到达。

In [ ]:
def years(value):
    return "未达到" if value is None else f"{value:.1f} 年"


breakthrough_advance = summary["feedback_advances_5pct_breakthrough_years"]
breakthrough_advance_50 = summary["feedback_advances_50pct_breakthrough_years"]
arrival_advance = summary["median_arrival_advance_years_where_both_arrive"]
top10_feedback = 100 * summary["final_top10_flow_share_feedback"]
top10_fixed = 100 * summary["final_top10_flow_share_fixed"]
verdict = summary["interpretation"]

rows = [
    {
        "指标": "5% 井端突破",
        "feedback": years(summary["feedback_5pct_breakthrough_years"]),
        "fixed": years(summary["fixed_5pct_breakthrough_years"]),
        "差异": (
            "无法比较"
            if breakthrough_advance is None
            else f"feedback 提前 {breakthrough_advance:.1f} 年"
        ),
    },
    {
        "指标": "50% 井端突破",
        "feedback": years(summary["feedback_50pct_breakthrough_years"]),
        "fixed": years(summary["fixed_50pct_breakthrough_years"]),
        "差异": (
            "无法比较"
            if breakthrough_advance_50 is None
            else f"feedback 提前 {breakthrough_advance_50:.1f} 年"
        ),
    },
    {
        "指标": (f"30 年 x={summary['flow_width_diagnostic_section_x_m']:.0f} m 断面有效过流宽度"),
        "feedback": f"{summary['final_effective_flow_width_feedback_m']:.1f} m",
        "fixed": f"{summary['final_effective_flow_width_fixed_m']:.1f} m",
        "差异": (f"feedback 缩窄 {summary['feedback_flow_width_reduction_percent']:.1f}%"),
    },
    {
        "指标": "30 年前 10% 单元流速份额",
        "feedback": f"{top10_feedback:.1f}%",
        "fixed": f"{top10_fixed:.1f}%",
        "差异": f"feedback 增加 {top10_feedback - top10_fixed:.1f} 个百分点",
    },
    {
        "指标": "共同到达区的中位提前时间",
        "feedback": "—",
        "fixed": "—",
        "差异": "无法比较" if arrival_advance is None else f"{arrival_advance:.1f} 年",
    },
    {
        "指标": "最大水力传导系数倍率",
        "feedback": f"{summary['maximum_K_multiplier_feedback']:.2f}",
        "fixed": "1.00",
        "差异": "反应造成的局部放大",
    },
    {
        "指标": "30 年井区平均水头",
        "feedback": f"{summary['final_mean_well_head_feedback_m']:.2f} m",
        "fixed": f"{summary['final_mean_well_head_fixed_m']:.2f} m",
        "差异": (
            f"feedback 高 "
            f"{summary['final_mean_well_head_feedback_m'] - summary['final_mean_well_head_fixed_m']:.2f} m"
        ),
    },
]
display(pd.DataFrame(rows).style.hide(axis="index"))
print("综合判断：", verdict)

## 1. 初始对数高斯 K 场

三层使用同一个三维相关随机场结构，并按各层几何平均水力传导系数缩放。直方图用于核对 ln(K/Kg) 近似正态。

In [ ]:
display(Image(filename=str(COMPARISON_DIR / "01_initial_log_gaussian_K.png")))

## 2. 通道随时间形成

第一排是反馈产生的水力传导系数倍率，第二排是补给水羽，第三排是相对 fixed 的流速放大。第三排若出现连续红色带，表示反馈后新增或增强的优势流动路径。

In [ ]:
display(Image(filename=str(COMPARISON_DIR / "02_channel_evolution.png")))

## 3. feedback 与 fixed 的空间差异

下排直接比较 5% 补给水到达时间。右下图为 fixed 到达时间减 feedback 到达时间：正值表示反馈使补给水提前到达。

In [ ]:
display(Image(filename=str(COMPARISON_DIR / "03_feedback_fixed_spatial_comparison.png")))

## 4. 四项通道指标

如果 feedback 曲线表现为突破更早、有效过流宽度更小、前 10% 单元流速份额更高，就说明流量正在从宽缓渗流转为少数优势路径控制。

In [ ]:
display(Image(filename=str(COMPARISON_DIR / "04_channel_metrics.png")))

## 5. 优势通道位于哪里

左图沿 x 方向比较每个断面的有效过流宽度；右图正值表示 feedback 的过流带比 fixed 更窄。它用于判断通道化主要发生在补水渠附近、模型中部还是井区。

In [ ]:
display(Image(filename=str(COMPARISON_DIR / "05_flow_width_profile.png")))